# Headgroup RDFs without classification (all DOPC lipids)

2D (xy) radial distribution functions between headgroup atoms, using **all** DOPC lipids (no machine-learning labels):

| curve | reference | target |
|---|---|---|
| CHOL(O)—DOPC(P) | DOPC P (atom 19) | CHOL O (atom 2) |
| CHOL(O)—DOPC(N) | DOPC N (atom 0) | CHOL O (atom 2) |
| DOPC(P)—DOPC(P) | DOPC P | DOPC P of the *other* lipids |
| DOPC(P)—DOPC(N) | DOPC P | DOPC N of the *other* lipids |

Same method as the NN headgroup RDFs: minimum-image xy distances, g(r) averaged over reference atoms in each frame,
then over frames, then over the two leaflets; box = time-averaged box. For P—P and P—N a lipid's own atom is left out
and the density uses N − 1, as in `counts_rdfs_dopc_dopc`. Positions are used frame by frame (`N_POINT = "raw"`).

Cell 3 is slow (reads the positions of each system); cell 4 only reads the saved CSV.

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

# ---- 1. import packages ----
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from code_libraries.classified_analysis import pbc_distances_xy, rdf_bin_edges, running_mean, unwrap_xy, load_leaflets

In [ ]:
# ---- 2. file paths and settings ----
SYSTEMS = {"dopc_90": 0.10, "dopc_75": 0.25, "dopc_65": 0.35}      # system -> chi_C (one panel each)
start_frame, stop_frame = 0, 4000
N_POINT = "raw"           # "raw" = positions frame by frame; or e.g. 5 = running average over 5 frames (with xy unwrapping)

P_ATOM, N_ATOM, CHOL_O_ATOM = 19, 0, 2
dr, cutoff = 0.05, 3.0    # bin width and maximum r (nm)

heavy_dir = "../heavy_data_NO_GIT"
out_dir   = os.path.join(heavy_dir, "results/hg_rdf")
os.makedirs(out_dir, exist_ok=True)
csv_file  = os.path.join(out_dir, f"no_ml_{'raw' if N_POINT == 'raw' else str(N_POINT) + 'pt'}_hg_rdf.csv")


def mean_box(sys_name):
    """Time-averaged box (x, y) from any classifier output for this system (the box does not depend on the model)."""
    files = sorted(glob.glob(os.path.join(heavy_dir, "data_arrays_by_chain_*", f"{sys_name}_sn1_*_box_dim.npy")))
    assert files, f"no box_dim file found for {sys_name}"
    box = np.load(files[0], allow_pickle=True)
    return np.asarray(box, dtype=float).reshape(-1, box.shape[-1]).mean(axis=0)[:2]


def rdf_xy(ref, tgt, box_xy, exclude_own=False, chunk=200):
    """2D g(r) of tgt around ref, averaged over ref atoms in each frame, then over frames.
    ref, tgt: (frames, n, 3). exclude_own: ref i and tgt i are the same lipid -> pair left out, density uses n_tgt - 1."""
    edges = rdf_bin_edges(dr, cutoff)
    shell = np.pi * (edges[1:] ** 2 - edges[:-1] ** 2)
    n_ref, n_tgt = ref.shape[1], tgt.shape[1]
    n_other = n_tgt - 1 if exclude_own else n_tgt
    density = n_other / (box_xy[0] * box_xy[1])
    g_sum, n_frames = np.zeros(len(shell)), 0
    for s in range(0, ref.shape[0], chunk):
        d = pbc_distances_xy(ref[s:s + chunk], tgt[s:s + chunk], box_xy)          # (frames, n_ref, n_tgt)
        if exclude_own:
            i = np.arange(min(n_ref, n_tgt))
            d[:, i, i] = np.inf
        for f in range(d.shape[0]):
            flat = np.sort(d[f].ravel())
            # strictly inside (inner, outer), as in the original shell counting
            counts = np.searchsorted(flat, edges[1:], side="left") - np.searchsorted(flat, edges[:-1], side="right")
            g_sum += counts / n_ref / (shell * density)
            n_frames += 1
    return g_sum / n_frames

In [ ]:
# ---- 3. calculate the RDFs (slow) ----
radial_distances = np.arange(0, cutoff, dr)
table = pd.DataFrame({"r_nm": radial_distances})

def positions(arr, atom, box_xy):
    pos = arr[:, :, atom, :]
    if N_POINT != "raw":
        pos = running_mean(unwrap_xy(pos, box_xy), int(N_POINT))
    return pos

for sys_name in SYSTEMS:
    box_xy = mean_box(sys_name)
    dopc_l, dopc_u, chol_l, chol_u = load_leaflets(sys_name, start_frame, stop_frame)
    g = {"O_P": [], "O_N": [], "P_P": [], "P_N": []}
    for dopc, chol in ((dopc_l, chol_l), (dopc_u, chol_u)):               # each leaflet with its own atoms
        P = positions(dopc, P_ATOM, box_xy)
        N = positions(dopc, N_ATOM, box_xy)
        O = positions(chol, CHOL_O_ATOM, box_xy)
        g["O_P"].append(rdf_xy(P, O, box_xy))
        g["O_N"].append(rdf_xy(N, O, box_xy))
        g["P_P"].append(rdf_xy(P, P, box_xy, exclude_own=True))
        g["P_N"].append(rdf_xy(P, N, box_xy, exclude_own=True))
    for pair, v in g.items():
        table[f"{sys_name}_{pair}"] = np.mean(v, axis=0)                  # average of the two leaflets
    print(f"{sys_name}: done (box {box_xy[0]:.2f} x {box_xy[1]:.2f} nm)", flush=True)
    del dopc_l, dopc_u, chol_l, chol_u

table.to_csv(csv_file, index=False)
print("saved", csv_file)

In [ ]:
# ---- 4. plot (reads the CSV saved in cell 3) ----
table = pd.read_csv(csv_file)

PAPER_STYLE = {"font.weight": "bold", "axes.labelweight": "bold", "axes.linewidth": 2.5, "mathtext.default": "bf",
               "xtick.top": True, "ytick.right": True, "xtick.direction": "in", "ytick.direction": "in",
               "xtick.major.size": 10, "ytick.major.size": 10, "xtick.major.width": 2.5, "ytick.major.width": 2.5,
               "xtick.minor.size": 5, "ytick.minor.size": 5, "xtick.minor.width": 1.8, "ytick.minor.width": 1.8}
D = "\u2014"     # long dash (em dash)
CURVES = [("O_P", "red",    f"CHOL(O){D}DOPC(P)"),
          ("O_N", "blue",   f"CHOL(O){D}DOPC(N)"),
          ("P_P", "green",  f"DOPC(P){D}DOPC(P)"),
          ("P_N", "purple", f"DOPC(P){D}DOPC(N)")]
X_MAX, Y_MAX = 2.0, 1.5
FS = 26           # tick labels, axis labels and chi_C label

with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, len(SYSTEMS), figsize=(9 * len(SYSTEMS), 8))
    for ax, (sys_name, chi) in zip(np.atleast_1d(axes), SYSTEMS.items()):
        for pair, color, label in CURVES:
            ax.plot(table["r_nm"], table[f"{sys_name}_{pair}"], color=color, lw=3, label=label)
        ax.text(0.95, 0.95, rf"$\mathbf{{\chi_C}}$ = {chi:g}", transform=ax.transAxes, fontsize=FS + 4,
                ha="right", va="top")
        ax.set_xlim(0, X_MAX)
        ax.set_ylim(0, Y_MAX)
        ax.set_xlabel("r (nm)", fontsize=FS)
        ax.set_ylabel("g(r)", fontsize=FS)
        ax.set_xticks(np.arange(0.5, X_MAX + 0.01, 0.5))       # x labels start at 0.5 (no clash with y = 0.0)
        ax.set_yticks(np.arange(0, Y_MAX + 0.01, 0.3))
        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))
        ax.tick_params(axis="both", which="major", labelsize=FS)
        ax.legend(fontsize=FS - 6, loc="lower right")
    plt.tight_layout(w_pad=3)
    plt.savefig(csv_file.replace(".csv", ".png"), dpi=300)
    plt.show()